## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\iarin\Desktop\ADC\inginerie AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [3]:
MY_BUBBLE_FILE = "personalist_salvator.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: personalist_salvator
Texte: 50


,id,agent,text
0,yt_X3bwh1-9nUU_Ugxc3Zx_bRhxFU18gXp4AaABAg,Personalist-salvator,"Ne au distrus hoții 😢,,,nu ne mai aparține nim..."
1,yt_bee6nXyzJ_E_UgyPoZoAIURM885eTr94AaABAg,Personalist-salvator,"Era si timpul sa treceti la atac, prea multa d..."
2,yt_bee6nXyzJ_E_UgzmsNNAxtBIZ34dyMB4AaABAg,Personalist-salvator,Vă mulțumim și noi și copii noștri care lucrea...
3,yt_bee6nXyzJ_E_UgzdyYPCNenomSY1CGN4AaABAg,Personalist-salvator,1:35 NOI știm că Sistemul ÎL hărțuiește mișele...
4,yt_bee6nXyzJ_E_UgwsuxSiNBqVBBSlPt14AaABAg,Personalist-salvator,Ideea că poporul are dreptul sau chiar datoria...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [4]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Ne au distrus hoții 😢,,,nu ne mai aparține nimic,,,, și au pus labele spurcate pe o țară in 89,,,,,,cu datorie externă 0,,,,,,,.


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [5]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\iarin\Desktop\ADC\inginerie AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iarin\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 2/2 [00:01<00:00,  1.79it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [6]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea fiecărui embedding (384 caracteristici).

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [7]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\personalist_salvator
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [6]:
# TODO student:
# index.faiss există: ...
# index.pkl există: ...
# index.ntotal este egal cu numărul de texte: ...

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [12]:
# Text nou introdus în aplicație

input_text = "Domnul Presedinte ales Calin Georgescu este singurul salvator al Romaniei, care poate sa scoata tara din criza in care a adus-o Klaus Iohannis. El este un om de o mare cultura, cu o experienta vasta in domeniul economic, care a demonstrat ca poate sa ia decizii rapide si eficiente in momentele de criza. Calin Georgescu este singurul candidat care poate sa aduca stabilitate si prosperitate Romaniei, si merita sa fie ales Presedinte."

In [13]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [14]:
# query_vector

In [15]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.585
Text: Dragul meu Robert,eu sunt un nimeni si poate nu cunosc așa bine cu ce se mănâncă politica dar in legătură cu dosarul domnului Georgescu, vineri seară cand am văzut acel protest la poarta Cotroceniului mi-am pus întrebarea, ce se va mai întâmpla in următoarele zile,eu am impresia că se aflase "pe surse" de rezultatul acelui dosar si de aia au ieșit. Iar în ceea ce-l privește pe Fritz părerea mea este că a vrut să arate încă o dată că ei conduc România...O zi bună tuturor.

Rezultat 2
Scor: 0.544
Text: In sfarsit cineva care pune punctul pe I direct. Cei mai rai oameni i-am vazut in biserica! Romania a devenit din democratie o teocratie corupta.

Rezultat 3
Scor: 0.532
Text: Doamne ajuta domnule Presedinte impreuna cu domnul Georgescu speram sa ne scapati odata de toti hoti din parlament sa avem si noi un trai ca in alte tari sa nu mai plecam ca tare greu este,apropo sunt ca mine care au cate 9 calificari in orice domeniu putem munci si in tara dar sa scapam

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

In [16]:
# Consider ca al doilea text este cel mai slab
# Cel mai bun text este al treilea